In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS pipeline_1.silver;

In [0]:
from datetime import datetime

# Задаем минимальную и текущую (максимальную) дату для чистки от неправильных данных по дате
MIN_DATE = '2025-01-01'
MAX_DATE = datetime.now().strftime('%Y-%m-%d')

BRONZE_TABLE = "pipeline_1.bronze.raw_events"
SILVER_TABLE = "pipeline_1.silver.silver_events"
CHECKPOINT_PATH = "/Volumes/pipeline_1/bronze/checkpoints_volume/silver_checkpoint"

In [0]:
from pyspark.sql.functions import explode, col, current_timestamp, substring

df_bronze = spark.readStream.table(BRONZE_TABLE)

# Разворачиваем массив из метрик, чтобы каждая метрика стояла отдельно
# Делаем конвертацию timestamp в корректный формат
# Фильтруем по датам, чтобы отсечь некорректные

df_exploded = (
    df_bronze
    .select(explode(col("values")).alias("metric_row"))
    .select(
        col("metric_row.device_id").alias("device_id"),
        col("metric_row.metric").alias("metric"),
        col("metric_row.value").alias("value"),
        (substring(col("metric_row.timestampUtc").cast("string"), 1, 13).cast("long") / 1000).cast("timestamp").alias("event_time")
    )
    .filter(f"event_time > '{MIN_DATE}' AND event_time < '{MAX_DATE}'")
)

In [0]:
def write_to_silver(microBatchDF, batchId):
    metrics = [
        "Wind_speed", "AirPres", "Ambient_temperature", "Humidity",
        "Power_output", "Rotor_rpm", "Error_code", "Brake_active",
        "Grid_connected", "Operation_mode"
    ]
    # Делаем пивот метрик, чтобы они отображались в колонках 
    # Кастуем данные в правильные типы
    # Убираем дубли через dropDuplicates - одно устройство в минуту
    # Используем append mode для инкрементальной загрузки данных
    (microBatchDF
        .groupBy("device_id", "event_time")
        .pivot("metric", metrics)
        .agg({"value": "first"})
        .withColumn("Wind_speed", col("Wind_speed").cast("float"))
        .withColumn("AirPres", col("AirPres").cast("float"))
        .withColumn("Ambient_temperature", col("Ambient_temperature").cast("float"))
        .withColumn("Humidity", col("Humidity").cast("float"))
        .withColumn("Power_output", col("Power_output").cast("float"))
        .withColumn("Rotor_rpm", col("Rotor_rpm").cast("int"))
        .withColumn("Error_code", col("Error_code").cast("int"))
        .withColumn("Brake_active", col("Brake_active").cast("boolean"))
        .withColumn("Grid_connected", col("Grid_connected").cast("boolean"))
        .withColumn("inserted_at", current_timestamp())
        .dropDuplicates(["device_id", "event_time"])
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_TABLE))

(df_exploded
    .writeStream
    .foreachBatch(write_to_silver)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start())